<div style="text-align:center; padding:24px 0 12px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="200"/>
</div>


# GoogleAdsPulse — Paid Media Analytics
## Notebook 1 — Contexte métier & Vue d'ensemble des KPIs

### 📝 **VERSION APPRENANT**

---

> **Objectif :** prendre en main le brief de Marc-Aurèle, découvrir les 5 tables,
> auditer la qualité, calculer les **10 KPIs ** et produire la viz Overview.

> **Ce notebook utilise uniquement pandas.** Le SQL avancé arrive au **NB2**.

| | |
|---|---|
| **Stack** | `pandas`, `matplotlib` |
| **Durée** | 2h30 à 3h30 |

---

### 📝 Comment utiliser ce notebook

- Les cellules **🎓 MÉTHODE** expliquent la démarche — lis-les attentivement
- Les cellules **📝 TODO** contiennent du code à compléter — suis les indices
- Les cellules **🧠 Tes observations** sont pour toi — décris en 3-5 lignes
- Pour vérifier : compare avec la solution ✅ une fois le code terminé


---
## 1. Contexte métier

### FluxData Agency — une agence digitale d'Abidjan

FluxData Agency est une agence digitale basée à Abidjan qui gère les comptes Google Ads
de **5 clients PME** d'Afrique de l'Ouest sur des secteurs variés : e-commerce
(TechShop CI), SaaS finance (BankAfrica SaaS), EdTech (FormaProPlus), hôtellerie
(AfriHotels Group) et santé B2B (MedSupply Pro). Budget cumulé géré : **~625 000 EUR**
sur 2023-2024.

### Le brief de Marc-Aurèle

> *« Chaque lundi, je passe 6 à 8 heures à consolider les reportings Google Ads de mes 5
> clients dans Excel. Je copie-colle les exports, je recalcule les deltas vs la semaine
> précédente, je refais les mêmes graphiques. C'est long, pas scalable, et mes clients
> veulent maintenant un accès direct à leurs KPIs.*
>
> *J'ai aussi du mal à savoir ce qui marche vraiment : Google Ads me donne un CPC flatteur
> sur certaines campagnes, mais quand je regarde le revenu généré, ce n'est pas forcément
> celles-là qui rapportent. Et avec 25 campagnes actives, je ne peux pas surveiller
> manuellement chaque dérapage budgétaire — il y a toujours une campagne qui consomme 3×
> son budget habituel sans que je le remarque avant la fin du mois.*
>
> *Il me faut un dashboard unifié qui centralise mes 5 comptes, affiche les KPIs avec leurs
> deltas, et détecte automatiquement les anomalies. »*

### 🎓 MÉTHODE — Traduire un brief verbal en questions analytiques

Marc-Aurèle exprime 3 préoccupations distinctes qu'il faut structurer en questions
analytiques concrètes :

| Phrase du brief | Question analytique | Livrable |
|---|---|---|
| *« Consolider les reportings de 5 clients chaque lundi »* | Centraliser les KPIs Google Ads standards dans une vue unifiée | NB1 — 10 KPIs  |
| *« Google Ads me donne un CPC flatteur, mais le revenu ne suit pas »* | Quelles campagnes ont le **meilleur ROAS** (revenu / spend) ? | NB2 — RANK() + NTILE |
| *« Une campagne consomme 3× son budget sans que je le remarque »* | Détecter automatiquement les **dérapages budgétaires** | NB2 — Z-score SQL |

### 🏢 MÉTIER — Ce qui différencie un Paid Media Manager senior

**1. Savoir lire au-delà du CPC affiché**

Les plateformes publicitaires optimisent leurs métriques pour paraître bonnes. Un CPC bas
peut cacher un trafic non qualifié qui ne convertit pas. Les vrais KPIs de pilotage sont
**Cost/Conversion** et **ROAS** (Return On Ad Spend = Conversions Value / Cost).

**2. Arbitrer Brand vs Generic**

Les campagnes **Brand** (le client tape le nom de l'annonceur) ont des CPC ultra-bas et
des conversions élevées — mais c'est du trafic qui serait venu quand même par l'organique.
Les campagnes **Generic** (mots-clés métier) coûtent plus cher mais génèrent du *vrai*
trafic incrémental. Bien arbitrer les deux est un vrai enjeu.

**3. Détecter les dérapages avant qu'ils ne coûtent**

Avec 25 campagnes actives, surveiller à l'œil est impossible. Un système d'alerte
automatique (z-score sur le spend quotidien) permet de repérer les jours anormaux
sans surcharge cognitive.


---
## 2. Dictionnaire des données — 5 tables

### Schéma en étoile

```
   accounts ──────► campaigns ──────► ads ──────► keywords
                        │               │             │
                        └───────────────┴─────────────┘
                                        │
                                        ▼
                          performance_quotidienne (fact)
                          granularité : date × campaign × ad × keyword
                                       × device × country × hour
```

### Détail des 5 tables

**`accounts.csv` (5 lignes)** — Comptes Google Ads gérés par l'agence

| Colonne | Type | Description |
|---|---|---|
| `account_id` | str | ACC001 → ACC005 |
| `account_name` | str | Nom commercial du client |
| `currency` | str | EUR (unifié pour tous les comptes) |
| `industry` | str | E-commerce, SaaS Finance, EdTech, Travel, B2B Health |
| `manager` | str | Marc-Aurèle T. ou Aïssatou D. |

**`campaigns.csv` (~25 lignes)** — Campagnes Google Ads

| Colonne | Type | Description |
|---|---|---|
| `campaign_id` | str | CAMP0001 → CAMP00XX |
| `campaign_name` | str | Nom du type `Brand-Search-CI`, `PerfMax-LeadGen`, etc. |
| `campaign_type` | str | Search / Display / Performance Max / Shopping / Video |
| `bid_strategy` | str | Manual CPC / Target CPA / Target ROAS / Maximize Conversions / Maximize Clicks |
| `budget_quotidien_eur` | int | Budget journalier planifié |
| `statut` | str | Enabled / Paused / Removed |
| `objectif` | str | Sales / Leads / Website traffic / Brand awareness |

**`ads.csv` (~120 lignes)** — Annonces avec Quality Score

| Colonne | Type | Description |
|---|---|---|
| `ad_id` | str | AD00001 → AD00XXX |
| `ad_group_name` | str | Nom de l'ad group parent |
| `headline_1`, `headline_2`, `description` | str | Copy publicitaire |
| `quality_score` | int | 1 à 10 — score de pertinence Google |

**`keywords.csv` (~370 lignes)** — Mots-clés (Search/Shopping)

| Colonne | Type | Description |
|---|---|---|
| `keyword_id` | str | KW00001 → KW00XXX |
| `keyword_text` | str | Texte du mot-clé |
| `match_type` | str | Exact / Phrase / Broad |
| `quality_score` | int | 1 à 10 |

**`performance_quotidienne.csv` (~41 400 lignes)** — Table de faits granulaire

| Colonne | Type | Description |
|---|---|---|
| `date` | date | Date du snapshot quotidien |
| `account_id`, `campaign_id`, `ad_id`, `keyword_id` | str | FKs |
| `device` | str | Desktop / Mobile / Tablet |
| `country`, `day_of_week`, `hour` | mixte | Contexte temporel et géo |
| `impressions`, `clicks`, `cost_eur` | int/float | **3 métriques de base** |
| `conversions`, `conversion_value_eur` | int/float | **Résultat** |
| `conversion_type` | str | Purchase / Lead / Signup / Demo Request / Form Submit / Phone Call |
| `avg_position` | float | Position moyenne dans les SERPs |

### ⚠️ Points d'attention dès la lecture du dictionnaire

| Piège | Impact | Vérification |
|---|---|---|
| `keyword_id` NULL pour les campaigns Display/Video/PMax | Normal (automated) | LEFT JOIN obligatoire |
| `quality_score` doit être dans [1-10] | Scores hors plage = bug ETL | Diagnostic à faire |
| `clicks > impressions` : impossible logiquement | Bug d'attribution | Diagnostic à faire |
| `cost_eur` doit être ≥ 0 | Les cost négatifs = bug de refund | Diagnostic à faire |
| `conversions > clicks` | Rare mais possible (multi-touch) | À flagger |


---
## 3. Setup & chargement

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

# Palette DataProjectLab — GoogleAdsPulse (style )
COLORS = {
    'primary':   '#534AB7',  # Violet DPL
    'secondary': '#1D9E75',  # Vert
    'warning':   '#EF9F27',  # Orange
    'danger':    '#E24B4A',  # Rouge
    'neutral':   '#888780',
    'blue':      '#0EA5E9',  # Bleu Coupler
}
CAMPAIGN_COLORS = {
    'Search':          '#4285F4',  # Google blue
    'Shopping':        '#34A853',  # Google green
    'Performance Max': '#9333EA',  # Violet
    'Display':         '#FBBC04',  # Google yellow
    'Video':           '#EA4335',  # Google red
}

print(f'✅ Environnement prêt — pandas {pd.__version__}')


### Chargement des donnees


In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/data/'

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/googleadspulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')


In [ ]:
accounts  = pd.read_csv(BASE_URL + 'accounts.csv', parse_dates=['date_onboarding'])
campaigns = pd.read_csv(BASE_URL + 'campaigns.csv', parse_dates=['start_date','end_date'])

ads        = pd.read_csv(BASE_URL + 'ads.csv')
keywords   = pd.read_csv(BASE_URL + 'keywords.csv')
perf       = pd.read_csv(BASE_URL + 'performance_quotidienne.csv', parse_dates=['date'])

# TODO — récap tableau


---
## 4. Exploration avec `pandas`

### 🎓 MÉTHODE — Routine d'exploration

Pour chaque table clé, 4 méthodes essentielles :

| Méthode | Révèle |
|---|---|
| `.head(n)` | Format visuel, structure brute |
| `.info()` | Types + nulls + mémoire |
| `.describe()` | Stats descriptives — détection d'outliers |
| `.value_counts()` | Distribution catégorielles — valeurs aberrantes |


In [ ]:
# 📝 TODO — Exploration de la table performance_quotidienne
#
# 1. Affiche les 3 premières lignes avec .head(3)
# 2. Affiche la structure avec .info() (types + nulls)

print('=' * 60)
print('  EXPLORATION — performance_quotidienne (table de faits)')
print('=' * 60)

# TODO — .head(3)

# TODO — .info()


In [ ]:
# 📝 TODO — Statistiques descriptives
#
# Utilise .describe() sur les 6 colonnes numériques :
# impressions, clicks, cost_eur, conversions, conversion_value_eur, avg_position
# 💡 Indice : perf[['col1','col2',...]].describe().round(2)

print('[.describe()] Métriques principales :')
# TODO


In [ ]:
# 📝 TODO — Distribution des variables catégorielles
#
# Pour chaque variable, affiche le décompte absolu ET le pourcentage normalisé
# Variables : device, country, conversion_type
# 💡 Indice : .value_counts() et .value_counts(normalize=True)

print('Distribution DEVICE :')
# TODO

print('\nDistribution COUNTRY :')
# TODO

print('\nDistribution CONVERSION_TYPE :')
# TODO


In [ ]:
# 📝 TODO — Exploration de la table campaigns (25 lignes)
#
# Affiche la distribution par : campaign_type, statut, objectif
# Puis la somme totale de budget_quotidien_eur

print('=' * 60)
print('  EXPLORATION — campaigns (25 lignes)')
print('=' * 60)

# TODO


In [ ]:
# 📝 TODO — Exploration de la table keywords (~370 lignes)
#
# Affiche : nombre total, distribution match_type, distribution quality_score,
# et le quality score moyen

print('=' * 60)
print('  EXPLORATION — keywords')
print('=' * 60)

# TODO


### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## 5. Diagnostic qualité

### 🎓 MÉTHODE — Audit en entonnoir

```
1. Structure (shape, types)       → Forme attendue ?
   ▼
2. Nulls                          → Qui manque, est-ce légitime ?
   ▼
3. Doublons                       → Clé primaire unique ?
   ▼
4. Anomalies techniques           → Valeurs impossibles (cost<0, clicks>imp)
   ▼
5. Incohérences métier            → QS hors plage, conversions>clicks
```


In [ ]:
# 📝 TODO — Audit des valeurs manquantes (nulls)
#
# Pour chaque table critique, vérifier les nulls sur les colonnes essentielles :
# - campaigns : ['campaign_id', 'account_id', 'campaign_type', 'statut']
# - ads       : ['ad_id', 'campaign_id', 'quality_score']
# - keywords  : ['keyword_id', 'campaign_id', 'quality_score']
# - perf      : ['date', 'account_id', 'campaign_id', 'device', 'impressions', 'clicks', 'cost_eur']
#
# 💡 Indice : df[col].isnull().sum() + flag '✅' si 0 null, '⚠️' sinon
# ⚠️ Note : keyword_id NULL dans perf est NORMAL (Display/Video/PMax = automated)

print('━' * 60)
print('  AUDIT QUALITÉ — GoogleAdsPulse')
print('━' * 60)

# TODO


In [ ]:
# 📝 TODO — Détection des doublons
#
# 1. Pour les tables à PK simple : df[pk].duplicated().sum()
# 2. Pour perf (pas de PK simple), clé composite :
#    ['date', 'campaign_id', 'ad_id', 'keyword_id', 'device', 'country', 'hour']

print('\n📋 2. DOUBLONS')
print('-' * 60)

# TODO


In [ ]:
# 📝 TODO — Détection des anomalies techniques
#
# Compter les occurrences des 5 anomalies :
# 1. cost_eur < 0 (impossible)
# 2. clicks > impressions (impossible)
# 3. conversions > clicks (rare mais possible en multi-touch)
# 4. conversions > 0 avec clicks = 0 (impossible)
# 5. quality_score hors [1-10] sur ads ET keywords
#
# 💡 (perf['col'] < 0).sum() et ((cond1) & (cond2)).sum()

print('\n📋 3. ANOMALIES TECHNIQUES')
print('-' * 60)

# TODO


In [ ]:
# 📝 TODO — Détection des incohérences métier (orphelins)
#
# 1. Ads orphelins (campaign_id absent de campaigns)
# 2. Keywords orphelins
# 3. Perf orphelines
# 4. Clicks > 0 avec cost_eur = 0 (suspect)
#
# 💡 camp_ids = set(campaigns['campaign_id']) puis (~ads['campaign_id'].isin(camp_ids)).sum()

print('\n📋 4. INCOHÉRENCES MÉTIER')
print('-' * 60)

# TODO


### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## 6. Les KPIs 

### 🎓 MÉTHODE — Pourquoi ces 10 KPIs ?

Ces 10 indicateurs forment la **vue Overview standard** d'un dashboard Google Ads
(reproduit par tous les outils pros : , Supermetrics, Looker Studio).
Ils se lisent en 3 groupes :

**Groupe 1 — Volume** (à quel volume tournons-nous ?)
| # | KPI | Calcul |
|---|---|---|
| 1 | **Impressions** | Nb de fois où l'annonce a été affichée |
| 2 | **Clicks** | Nb de clics reçus |
| 3 | **Conversions** | Nb d'actions valorisées (purchase, lead, etc.) |

**Groupe 2 — Efficacité** (à quel coût ?)
| # | KPI | Calcul | Règle |
|---|---|---|---|
| 4 | **CTR** | clicks / impressions × 100 | Plus élevé = mieux |
| 5 | **CPC** | cost / clicks | Plus bas = mieux |
| 6 | **CPM** | cost / impressions × 1000 | Plus bas = mieux |
| 7 | **Cost/Conversion** | cost / conversions | Plus bas = mieux |
| 8 | **Conversion rate** | conversions / clicks × 100 | Plus élevé = mieux |

**Groupe 3 — Rentabilité** (est-ce que ça rapporte ?)
| # | KPI | Calcul |
|---|---|---|
| 9 | **Spend amount** | somme des cost_eur |
| 10 | **Conversions value** | somme des conversion_value_eur |
| Bonus | **ROAS** | conversions value / spend (doit être > 3 pour rentabilité) |

> **⚠️ On reste sur des KPIs GLOBAUX.** Les deltas period-over-period, Top/Bottom,
> breakdowns et cohort analysis sont traités dans le **NB2** avec SQL.


In [ ]:
# 📝 TODO — Nettoyage minimal pour des KPIs fiables
#
# Créer perf_clean en enchaînant :
# 1. .drop_duplicates(subset=['date','campaign_id','ad_id','keyword_id','device','country','hour'])
# 2. .query('cost_eur >= 0')
# 3. .query('clicks <= impressions')
# 4. .copy()

# TODO
perf_clean = (perf
    # TODO — chainer les transformations
    .copy()
)

# TODO — afficher volume avant/après/% exclu


In [ ]:
# 📝 TODO — Calcul des 10 KPIs 
#
# GROUPE 1 — VOLUME (3 KPIs)
#   total_impressions, total_clicks, total_conversions (SUM)
#
# GROUPE 2 — EFFICACITÉ (5 KPIs ratios, protège division par 0)
#   ctr = clicks/impressions * 100
#   cpc = spend/clicks
#   cpm = spend/impressions * 1000
#   cost_per_conv = spend/conversions
#   conv_rate = conversions/clicks * 100
#
# GROUPE 3 — RENTABILITÉ (2 KPIs + bonus)
#   total_spend (SUM cost_eur)
#   total_conv_value (SUM conversion_value_eur)
#   roas = conv_value/spend
#
# ⚠️ Protège chaque ratio : 'X / Y if Y else 0'
# Affichage style  avec séparateurs '═' '─'

total_impressions = 0  # TODO
total_clicks      = 0  # TODO
total_conversions = 0  # TODO
total_spend       = 0  # TODO
total_conv_value  = 0  # TODO

ctr           = 0  # TODO
cpc           = 0  # TODO
cpm           = 0  # TODO
cost_per_conv = 0  # TODO
conv_rate     = 0  # TODO
roas          = 0  # TODO

periode_debut = perf_clean['date'].min().date()
periode_fin   = perf_clean['date'].max().date()
nb_jours      = (perf_clean['date'].max() - perf_clean['date'].min()).days + 1

# TODO — Affichage style  avec les 3 groupes


### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## 7. Visualisation Overview 2×2

### 🎓 MÉTHODE — Reproduire la page Overview

| Cadran | Graphique  | Type |
|---|---|---|
| Haut-gauche | Impressions vs Clicks | Dual-axis timeline |
| Haut-droit | Clicks vs CPC | Dual-axis timeline |
| Bas-gauche | Spend amount by Date | Single-line timeline |
| Bas-droit | Conversions vs Conversion rate | Dual-axis timeline |

Les 4 graphiques utilisent l'agrégation **quotidienne** lissée sur 7 jours pour la lisibilité.


In [ ]:
# 📝 TODO — Préparation de la série temporelle quotidienne
#
# 1. Grouper perf_clean par date, agréger impressions/clicks/cost/conversions (SUM)
# 2. Calculer :
#    - cpc = cost / clicks.replace(0, np.nan)
#    - conv_rate = conversions / clicks.replace(0, np.nan) * 100
# 3. Lissage mobile 7j avec .rolling(window=7, min_periods=1).mean()
#    sur les 6 colonnes → stocker dans {col}_ma7

# TODO
daily = None

print('✅ Série temporelle prête')


In [ ]:
# 📝 TODO — Page Overview (4 timelines 2×2)
#
# fig, axes = plt.subplots(2, 2, figsize=(16, 10))
#
# Cadran [0,0] — Impressions vs Clicks (dual-axis)
#   ax.plot(daily['date'], daily['impressions_ma7'], color=COLORS['primary'])
#   ax.twinx().plot(daily['date'], daily['clicks_ma7'], color=COLORS['blue'])
#
# Cadran [0,1] — Clicks vs CPC (bleu + orange COLORS['warning'])
# Cadran [1,0] — Spend by Date (ligne rouge + fill_between alpha=0.12)
# Cadran [1,1] — Conversions vs Conv. rate (vert + violet '#9333EA')
#
# 💡 FuncFormatter pour axe Y en K
# 💡 tick_params(axis='x', rotation=30), tight_layout()
# 💡 plt.savefig(SAVE_PATH + 'googleadspulse_overview.png', dpi=150)

# TODO


### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## 8. Bilan du Notebook 1

### ✅ Ce qui a été réalisé

| Étape | Livrable |
|---|---|
| Brief métier | 3 questions de Marc-Aurèle traduites en questions analytiques |
| Dictionnaire | 5 tables documentées + schéma étoile |
| Chargement | 5 DataFrames pandas avec `parse_dates` |
| Exploration | `.head()`, `.info()`, `.describe()`, `.value_counts()` sur les 3 tables clés |
| Diagnostic | Nulls, doublons, anomalies techniques, incohérences métier |
| 10 KPIs  | Volume + Efficacité + Rentabilité |
| Visualisation | Figure 2×2 Overview (style ) |

### 📋 Checklist pour le NB2

```
☐  Réutiliser perf_clean (dédoublonné + filtré)
☐  Calculer les 10 KPIs EN SQL avec deltas period-over-period (LAG)
☐  Top & Bottom campaigns avec RANK + NTILE
☐  Rolling 7-day average sur les KPIs clés (SUM OVER ROWS BETWEEN)
☐  Cohort analysis des keywords (DATE_TRUNC + DATEDIFF + pivot)
☐  Détection anomalies par z-score (AVG OVER + STDDEV OVER)
☐  Breakdown device × jour × heure (pivots SQL)
☐  Self-join campagne vs benchmark account
☐  Export des 7 CSV analytiques pour Power BI
```

### 🧭 Progression pédagogique

```
NB1 ──► Contexte + 10 KPIs Overview (ce notebook)
 │
 ▼
NB2 ──► SQL avancé : window functions + cohort analysis + z-score
 │
 ▼
NB3 ──► Power BI : 5 pages dashboard style  + 30+ mesures DAX
```


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.
